In [16]:
!pip install duckdb

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ---------------------------------------- 0.1/13.2 MB 656.4 kB/s eta 0:00:20
    --------------------------------------- 0.3/13.2 MB 2.2 MB/s eta 0:00:06
   -- ------------------------------------- 0.7/13.2 MB 3.8 MB/s eta 0:00:04
   --- ------------------------------------ 1.1/13.2 MB 5.1 MB/s eta 0:00:03
   ---- ----------------------------------- 1.5/13.2 MB 5.8 MB/s eta 0:00:03
   ------ --------------------------------- 2.1/13.2 MB 7.0 MB/s eta 0:00:02
   ------- -------------------------------- 2.6/13.2 MB 7.5 MB/s eta 0:00:02
   --------- ------------------------------ 3.2/13.2 MB 8.1 MB/s eta 0:00:02
   ----------- ---------------------------- 3.7/13.2 MB 8.4 MB/s eta 0:00:02
   ------------ --------------------------- 4.2/13.2 MB 8.6 MB/s eta 0:00:02
   -------------- ------------------------- 4.7/13.2 MB 8.9 MB/s eta 0:00:01
   --------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 00 — Data Discovery: Procurement Intelligence Platform

**Fase 1 do projeto.** Objetivo: validar com dados reais se compras.gov.br / PNCP
suportam o Módulo 2 (Price Intelligence) do projeto, antes de comprometer a
arquitetura Bronze/Silver/Gold definida na Fase 0.

**Princípio metodológico:** não presumir nada que este notebook não confirme.
Cada seção termina em uma métrica objetiva, não em impressão.

**Como usar:** rode célula a célula. Ao final, preencha a Seção 11 (checklist
de decisão GO / GO COM AJUSTES / NO-GO) com os números que você observou e
traga o notebook executado de volta para revisarmos juntos.

**Referência:** *Manual do Usuário – API do Compras.gov.br*, versão 2.0 (Fev/2026),
`https://www.gov.br/compras/pt-br/acesso-a-informacao/manuais/manual-dados-abertos/manual-api-compras.pdf`


In [1]:
# Setup
import requests
import pandas as pd
import numpy as np
import time
import json
from pathlib import Path
from datetime import date, timedelta

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

BASE_API = "https://dadosabertos.compras.gov.br"
BASE_BULK = "https://repositorio.dados.gov.br/seges/comprasgov"

DATA_DIR = Path("../data/discovery")
DATA_DIR.mkdir(parents=True, exist_ok=True)

SESSION = requests.Session()
SESSION.headers.update({"accept": "application/json"})

def get_json(url, params=None, timeout=30):
    """GET com retry simples e log do que foi de fato chamado."""
    for attempt in range(3):
        try:
            resp = SESSION.get(url, params=params, timeout=timeout)
            print(f"GET {resp.url}  ->  status {resp.status_code}")
            if resp.status_code == 200:
                return resp.json()
            else:
                print("Corpo da resposta (para diagnóstico):")
                print(resp.text[:2000])
                return None
        except requests.exceptions.RequestException as e:
            print(f"Tentativa {attempt+1} falhou: {e}")
            time.sleep(2 * (attempt + 1))
    return None


## Seção 1 — Módulo Contratações: Contratações PNCP 14133 (nível compra)

Testa o endpoint `1_consultarContratacoes_PNCP_14133`. Parâmetros
`dataPublicacaoPncpInicial`, `dataPublicacaoPncpFinal` e `codigoModalidade`
são documentados como obrigatórios — vamos confirmar isso na prática.

Usamos uma janela curta (últimos 30 dias) e `codigoModalidade=05` (Pregão,
a modalidade mais comum) só para validar o schema real, não para ingestão.


In [2]:
# Janela curta só para inspecionar o schema real de resposta
hoje = date.today()
inicio = hoje - timedelta(days=30)

params_contratacoes = {
    "pagina": 1,
    "tamanhoPagina": 50,
    "dataPublicacaoPncpInicial": inicio.strftime("%Y-%m-%d"),
    "dataPublicacaoPncpFinal": hoje.strftime("%Y-%m-%d"),
    "codigoModalidade": 6,  # 06 = Dispensa de Licitação, costuma ter volume alto e resposta rápida
}

url_contratacoes = f"{BASE_API}/modulo-contratacoes/1_consultarContratacoes_PNCP_14133"
resp = get_json(url_contratacoes, params=params_contratacoes)

if resp:
    print("\nTotal de registros na janela:", resp.get("totalRegistros"))
    df_contratacoes = pd.json_normalize(resp.get("resultado", []))
    display(df_contratacoes.head(10))
    print("\nColunas retornadas:")
    print(list(df_contratacoes.columns))
else:
    df_contratacoes = pd.DataFrame()
    print("Chamada falhou — ver diagnóstico acima antes de prosseguir.")


GET https://dadosabertos.compras.gov.br/modulo-contratacoes/1_consultarContratacoes_PNCP_14133?pagina=1&tamanhoPagina=50&dataPublicacaoPncpInicial=2026-07-11&dataPublicacaoPncpFinal=2026-08-10&codigoModalidade=6  ->  status 200

Total de registros na janela: 4704


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,orgaoSubrogadoEsferaId,orgaoEntidadePoderId,orgaoSubrogadoPoderId,unidadeOrgaoCodigoUnidade,unidadeSubrogadaCodigoUnidade,unidadeOrgaoNomeUnidade,unidadeSubrogadaNomeUnidade,unidadeOrgaoUfSigla,unidadeSubrogadaUfSigla,unidadeOrgaoMunicipioNome,unidadeSubrogadaMunicipioNome,unidadeOrgaoCodigoIbge,unidadeSubrogadaCodigoIbge,numeroCompra,modalidadeIdPncp,codigoModalidade,modalidadeNome,srp,modoDisputaIdPncp,codigoModoDisputa,amparoLegalCodigoPncp,amparoLegalNome,amparoLegalDescricao,informacaoComplementar,processo,objetoCompra,existeResultado,orcamentoSigilosoCodigo,orcamentoSigilosoDescricao,situacaoCompraIdPncp,situacaoCompraNomePncp,tipoInstrumentoConvocatorioCodigoPncp,tipoInstrumentoConvocatorioNome,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida
0,15306306004762026,34621748000123-1-000220/2026,2026,220,34621748000123,None,80832,UNIVERSIDADE FEDERAL DO PARA,None,F,None,E,None,153063,None,UNIVERSIDADE FEDERAL DO PARA/PA,None,PA,None,BELÉM,None,1501402,None,476,8,6,Dispensa,False,4,4,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que en...,Informamos que a descrição dos itens e unidade...,23073.052269/2026-61,"AQUISIÇÃO DE TONERS ORIGINAIS PARA CPGA, PROTO...",False,1,Compra sem sigilo,1,Divulgada no PNCP,2,Aviso de Contratação Direta,Dispensa Com Disputa,4076.59,NaN,2026-07-11T07:26:46,2026-07-11T07:26:46,2026-07-11T07:26:46,2026-07-13T08:00:00,2026-07-16T08:00:00,False
1,07001306000812026,00509018000113-1-001503/2026,2026,1503,00509018000113,None,47412,TRIBUNAL SUPERIOR ELEITORAL,None,F,None,J,None,070013,None,TRIBUNAL REGIONAL ELEITORAL DA BAHIA,None,BA,None,SALVADOR,None,2927408,None,81,8,6,Dispensa,False,4,4,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que en...,NaN,0009276-59.2026.6.05.8000,Contratação de empresa especializada em produç...,False,1,Compra sem sigilo,1,Divulgada no PNCP,2,Aviso de Contratação Direta,Dispensa Com Disputa,31730.92,NaN,2026-07-11T12:01:20,2026-07-13T08:45:47,2026-07-11T12:01:20,2026-07-13T10:00:00,2026-07-16T08:00:00,False
2,78432006000182026,00394502000144-1-007704/2026,2026,7704,00394502000144,None,46041,COMANDO DA MARINHA,None,F,None,E,None,784320,None,CAPITANIA DOS PORTOS DO ESTADO DO MARANHAO,None,MA,None,SÃO LUÍS,None,2111300,None,18,8,6,Dispensa,False,4,4,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que en...,Atenção ao local de entrega quanto aos itens s...,63036.001507/2026-31,Contratação de serviço de confecção de crachás...,False,1,Compra sem sigilo,1,Divulgada no PNCP,2,Aviso de Contratação Direta,Dispensa Com Disputa,1790.00,NaN,2026-07-11T19:22:27,2026-07-11T19:22:27,2026-07-11T19:22:27,2026-07-13T08:00:00,2026-07-16T08:00:00,False
3,78120006000372026,00394502000144-1-007705/2026,2026,7705,00394502000144,None,46041,COMANDO DA MARINHA,None,F,None,E,None,781200,None,GRUPAMENTO DE FUZILEIROS NAVAIS_DO RJ,None,RJ,None,RIO DE JANEIRO,None,3304557,None,37,8,6,Dispensa,False,4,4,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação: para contratação que en...,A aquisição visa prover a continuidade e a efi...,63190.001952/2026-18,Aquisição de equipamentos de informática.,True,1,Compra sem sigilo,1,Divulgada no PNCP,2,Aviso de Contratação Direta,Dispensa Com Disputa,2238.36,1707.00,2026-07-11T20:41:16,2026-07-11T20:41:16,2026-07-11T20:41:16,2026-07-13T08:00:00,2026-07-16T08:00:00,False
4,78120006000442026,00394502000144-1-007706/2026,2026,7706,00394502000144,None,46041,COMANDO DA MARINHA,None,F,None,E,None,781200,None,GRUPAMENTO DE FUZILEIROS NAVAIS_DO RJ,None,RJ,None,RIO DE JANEIRO,None,3304557,None,44,8,6,Dispensa,False,4,4,19,"Lei 14.133/2021, Art. 75, II",Dispensa de Licitação


Colunas retornadas:
['idCompra', 'numeroControlePNCP', 'anoCompraPncp', 'sequencialCompraPncp', 'orgaoEntidadeCnpj', 'orgaoSubrogadoCnpj', 'codigoOrgao', 'orgaoEntidadeRazaoSocial', 'orgaoSubrogadoRazaoSocial', 'orgaoEntidadeEsferaId', 'orgaoSubrogadoEsferaId', 'orgaoEntidadePoderId', 'orgaoSubrogadoPoderId', 'unidadeOrgaoCodigoUnidade', 'unidadeSubrogadaCodigoUnidade', 'unidadeOrgaoNomeUnidade', 'unidadeSubrogadaNomeUnidade', 'unidadeOrgaoUfSigla', 'unidadeSubrogadaUfSigla', 'unidadeOrgaoMunicipioNome', 'unidadeSubrogadaMunicipioNome', 'unidadeOrgaoCodigoIbge', 'unidadeSubrogadaCodigoIbge', 'numeroCompra', 'modalidadeIdPncp', 'codigoModalidade', 'modalidadeNome', 'srp', 'modoDisputaIdPncp', 'codigoModoDisputa', 'amparoLegalCodigoPncp', 'amparoLegalNome', 'amparoLegalDescricao', 'informacaoComplementar', 'processo', 'objetoCompra', 'existeResultado', 'orcamentoSigilosoCodigo', 'orcamentoSigilosoDescricao', 'situacaoCompraIdPncp', 'situacaoCompraNomePncp', 'tipoInstrumentoConvocatorioC

In [3]:
# Teste de robustez: a API realmente exige codigoModalidade e as datas?
# (documentação marca como obrigatório — vamos confirmar o comportamento real)
params_sem_modalidade = {k: v for k, v in params_contratacoes.items() if k != "codigoModalidade"}
print("Chamando SEM codigoModalidade:")
resp_sem_mod = get_json(url_contratacoes, params=params_sem_modalidade)


Chamando SEM codigoModalidade:
GET https://dadosabertos.compras.gov.br/modulo-contratacoes/1_consultarContratacoes_PNCP_14133?pagina=1&tamanhoPagina=50&dataPublicacaoPncpInicial=2026-07-11&dataPublicacaoPncpFinal=2026-08-10  ->  status 404
Corpo da resposta (para diagnóstico):
{ "statusCode": 404, "message": "Resource not found" }


## Seção 2 — Itens das Contratações (nível item)

Testa `2_consultarItensContratacoes_PNCP_14133`. A documentação lista
`materialOuServico`, `codigoClasse` e `codigoGrupo` como obrigatórios, mas as
descrições desses parâmetros no manual oficial parecem trocadas (texto de
outra seção) — típico de erro de geração de PDF. Vamos testar o comportamento
real em vez de confiar cegamente no texto.


In [4]:
url_itens = f"{BASE_API}/modulo-contratacoes/2_consultarItensContratacoes_PNCP_14133"

# Tentativa 1: só com materialOuServico (o único parâmetro cujo nome faz sentido como obrigatório)
params_itens_v1 = {
    "pagina": 1,
    "tamanhoPagina": 50,
    "materialOuServico": "M",
}
print("Tentativa 1 — apenas materialOuServico:")
resp_itens_v1 = get_json(url_itens, params=params_itens_v1)


Tentativa 1 — apenas materialOuServico:
GET https://dadosabertos.compras.gov.br/modulo-contratacoes/2_consultarItensContratacoes_PNCP_14133?pagina=1&tamanhoPagina=50&materialOuServico=M  ->  status 404
Corpo da resposta (para diagnóstico):
{ "statusCode": 404, "message": "Resource not found" }


In [5]:
# Tentativa 2: adicionando codigoGrupo e codigoClasse de um grupo de material comum
# (ex.: grupo 32 = Material de Escritório, ajuste conforme o que a Tentativa 1 revelar)
params_itens_v2 = {
    "pagina": 1,
    "tamanhoPagina": 50,
    "materialOuServico": "M",
    "codigoGrupo": 32,
    "codigoClasse": 3210,
}
print("Tentativa 2 — com codigoGrupo/codigoClasse:")
resp_itens_v2 = get_json(url_itens, params=params_itens_v2)

resp_itens = resp_itens_v2 if resp_itens_v2 else resp_itens_v1
if resp_itens:
    df_itens = pd.json_normalize(resp_itens.get("resultado", []))
    display(df_itens.head(10))
    print("\nColunas retornadas:")
    print(list(df_itens.columns))
else:
    df_itens = pd.DataFrame()
    print("Nenhuma das duas tentativas funcionou — registrar o erro exato para decidirmos o próximo passo.")


Tentativa 2 — com codigoGrupo/codigoClasse:
GET https://dadosabertos.compras.gov.br/modulo-contratacoes/2_consultarItensContratacoes_PNCP_14133?pagina=1&tamanhoPagina=50&materialOuServico=M&codigoGrupo=32&codigoClasse=3210  ->  status 404
Corpo da resposta (para diagnóstico):
{ "statusCode": 404, "message": "Resource not found" }
Nenhuma das duas tentativas funcionou — registrar o erro exato para decidirmos o próximo passo.


**Achado confirmado nesta rodada:** as duas tentativas retornaram 404 —
não é erro de parâmetro, é o endpoint `2_consultarItensContratacoes_PNCP_14133`
não respondendo no caminho documentado (os endpoints 1 e 3 do mesmo módulo
funcionaram normalmente com o mesmo padrão de URL). Não vamos depender desse
endpoint: o CSV bulk baixado na Seção 4 já traz os campos de item e de
resultado mesclados numa única linha por `id_compra_item`, o que resolve
o mesmo problema por outro caminho.


## Seção 3 — Resultado dos Itens (preço efetivamente praticado)

Testa `3_consultarResultadoItensContratacoes_PNCP_14133`. Este é o endpoint
que traz `valorUnitarioHomologado` — o preço que de fato foi pago, não o
estimado. `dataResultadoPncpInicial`/`Final` são documentados como obrigatórios.


In [6]:
url_resultado = f"{BASE_API}/modulo-contratacoes/3_consultarResultadoItensContratacoes_PNCP_14133"

params_resultado = {
    "pagina": 1,
    "tamanhoPagina": 50,
    "dataResultadoPncpInicial": inicio.strftime("%Y-%m-%d"),
    "dataResultadoPncpFinal": hoje.strftime("%Y-%m-%d"),
}
resp_resultado = get_json(url_resultado, params=params_resultado)

if resp_resultado:
    print("\nTotal de registros na janela:", resp_resultado.get("totalRegistros"))
    df_resultado = pd.json_normalize(resp_resultado.get("resultado", []))
    display(df_resultado.head(10))
    print("\nColunas retornadas:")
    print(list(df_resultado.columns))
else:
    df_resultado = pd.DataFrame()
    print("Chamada falhou — ver diagnóstico acima.")


GET https://dadosabertos.compras.gov.br/modulo-contratacoes/3_consultarResultadoItensContratacoes_PNCP_14133?pagina=1&tamanhoPagina=50&dataResultadoPncpInicial=2026-07-11&dataResultadoPncpFinal=2026-08-10  ->  status 200

Total de registros na janela: 80275


,idCompraItem,idCompra,idContratacaoPNCP,unidadeOrgaoCodigoUnidade,unidadeOrgaoUfSigla,numeroItemPncp,sequencialResultado,niFornecedor,tipoPessoa,nomeRazaoSocialFornecedor,codigoPais,indicadorSubcontratacao,ordemClassificacaoSrp,quantidadeHomologada,valorUnitarioHomologado,valorTotalHomologado,percentualDesconto,situacaoCompraItemResultadoId,situacaoCompraItemResultadoNome,motivoCancelamento,porteFornecedorId,porteFornecedorNome,naturezaJuridicaNome,naturezaJuridicaId,dataInclusaoPncp,dataAtualizacaoPncp,dataCancelamentoPncp,dataResultadoPncp,numeroControlePNCPCompra,orgaoEntidadeCnpj,aplicacaoMargemPreferencia,amparoLegalMargemPreferenciaId,amparoLegalMargemPreferenciaNome,aplicacaoBeneficioMeepp,aplicacaoCriterioDesempate,amparoLegalCriterioDesempateId,amparoLegalCriterioDesempateNome,moedaEstrangeiraId,dataCotacaoMoedaEstrangeira,valorNominalMoedaEstrangeira,paisOrigemProdutoServicoId,timezoneCotacaoMoedaEstrangeira
0,1603280700083202600003,16032807000832026,00394452000103-1-013173/2026,160328,RJ,3,1,07471449000187,PJ,DAFRA TECHNOLOGIES INSTRUMENTACAO ANALITICA E ...,BRA,False,1.0,1.0,6370.0,6370.0,0.0,1,Informado,None,3,Demais,Sociedade Empresária Limitada,2062,2026-07-11T13:02:32,2026-07-11T13:02:32,None,2026-07-11T00:00:00,00394452000103-1-013173/2026,00394452000103,False,None,None,False,False,None,None,None,None,None,None,None
1,1603280700083202600002,16032807000832026,00394452000103-1-013173/2026,160328,RJ,2,1,07471449000187,PJ,DAFRA TECHNOLOGIES INSTRUMENTACAO ANALITICA E ...,BRA,False,1.0,1.0,6700.0,6700.0,0.0,1,Informado,None,3,Demais,Sociedade Empresária Limitada,2062,2026-07-11T13:02:32,2026-07-11T13:02:32,None,2026-07-11T00:00:00,00394452000103-1-013173/2026,00394452000103,False,None,None,False,False,None,None,None,None,None,None,None
2,1603280700083202600005,16032807000832026,00394452000103-1-013173/2026,160328,RJ,5,1,07471449000187,PJ,DAFRA TECHNOLOGIES INSTRUMENTACAO ANALITICA E ...,BRA,False,1.0,1.0,5255.0,5255.0,0.0,1,Informado,None,3,Demais,Sociedade Empresária Limitada,2062,2026-07-11T13:02:32,2026-07-11T13:02:32,None,2026-07-11T00:00:00,00394452000103-1-013173/2026,00394452000103,False,None,None,False,False,None,None,None,None,None,None,None
3,1603280700083202600004,16032807000832026,00394452000103-1-013173/2026,160328,RJ,4,1,07471449000187,PJ,DAFRA TECHNOLOGIES INSTRUMENTACAO ANALITICA E ...,BRA,False,1.0,1.0,4620.0,4620.0,0.0,1,Informado,None,3,Demais,Sociedade Empresária Limitada,2062,2026-07-11T13:02:32,2026-07-11T13:02:32,None,2026-07-11T00:00:00,00394452000103-1-013173/2026,00394452000103,False,None,None,False,False,None,None,None,None,None,None,None
4,1605550700259202600001,16055507002592026,00394452000103-1-013174/2026,160555,PI,1,1,11937056490,PF,ADMO JOSE DE CARVALHO NUNES,BRA,False,1.0,232.0,250.0,58000.0,0.0,1,Informado,None,5,Não Informado,NaN,NaN,2026-07-11T14:21:19,2026-07-11T14:21:19,None,2026-07-11T00:00:00,00394452000103-1-013174/2026,00394452000103,False,None,None,False,False,None,None,None,None,None,None,None
5,1605550700260202600001,16055507002602026,00394452000103-1-013175/2026,160555,PI,1,1,92424473315,PF,RAMIRO DIAS TEIXEIRA,BRA,False,1.0,232.0,250.0,58000.0,0.0,1,Informado,None,5,Não Informado,NaN,NaN,2026-07-11T14:24:51,2026-07-11T14:24:51,None,2026-07-11T00:00:00,00394452000103-1-013175/2026,00394452000103,False,None,None,False,False,None,None,None,None,None,None,None
6,1605550700261202600001,16055507002612026,00394452000103-1-013176/2026,160555,PI,1,1,74860267320,PF,CARLITO MANOEL DE SOUSA,BRA,False,1.0,232.0,250.0,58000.0,0.0,1,Informado,None,5,Não Informado,NaN,NaN,2026-07-11T14:30:00,2026-07-11T14:30:00,None,2026-07-11T00:00:00,00394452000103-1-013176/2026,00394452000103,False,None,None,False,False,None,None,None,None,None,None,None
7,1605550700262202600001,16055507002622026,00394452000103-1-013177/2026,160555,PI,1,1,05555494378,PF,JOSE LUCAS RODRIGUES,BRA,False,1.0,232.0,250.0,58000.0,0.0,1,Informado,None,5,Não Informado,NaN,NaN,2026-07-11T14:33:23,2026-07-11T14:33:23,None,


Colunas retornadas:
['idCompraItem', 'idCompra', 'idContratacaoPNCP', 'unidadeOrgaoCodigoUnidade', 'unidadeOrgaoUfSigla', 'numeroItemPncp', 'sequencialResultado', 'niFornecedor', 'tipoPessoa', 'nomeRazaoSocialFornecedor', 'codigoPais', 'indicadorSubcontratacao', 'ordemClassificacaoSrp', 'quantidadeHomologada', 'valorUnitarioHomologado', 'valorTotalHomologado', 'percentualDesconto', 'situacaoCompraItemResultadoId', 'situacaoCompraItemResultadoNome', 'motivoCancelamento', 'porteFornecedorId', 'porteFornecedorNome', 'naturezaJuridicaNome', 'naturezaJuridicaId', 'dataInclusaoPncp', 'dataAtualizacaoPncp', 'dataCancelamentoPncp', 'dataResultadoPncp', 'numeroControlePNCPCompra', 'orgaoEntidadeCnpj', 'aplicacaoMargemPreferencia', 'amparoLegalMargemPreferenciaId', 'amparoLegalMargemPreferenciaNome', 'aplicacaoBeneficioMeepp', 'aplicacaoCriterioDesempate', 'amparoLegalCriterioDesempateId', 'amparoLegalCriterioDesempateNome', 'moedaEstrangeiraId', 'dataCotacaoMoedaEstrangeira', 'valorNominalMoed

**Correção à matriz de campos original:** `orgaoEntidadeCnpj` e
`unidadeOrgaoUfSigla` vieram diretamente nesta resposta da API — não precisam
de join *quando a fonte é este endpoint*. Isso não vale para o CSV bulk de
item (Seção 4): lá esses dois campos não aparecem, e o join com
`VW_FT_PNCP_COMPRA` continua necessário para o pipeline baseado em CSV.
A Seção 10 já reflete essa distinção.


## Seção 4 — Amostra via arquivos bulk CSV

Para ingestão histórica (2022-2025), API paginada não é viável no volume
necessário. Testamos aqui um arquivo **diário** (pequeno) do repositório de
dados abertos para validar formato, separador decimal e nomes de coluna reais
— inclusive o problema de casas decimais relatado por usuários no fórum oficial.

Ajuste a data abaixo para um dia recente que você saiba que teve movimento
(qualquer dia útil recente deve funcionar).


In [7]:
# Ajuste a data conforme necessário
data_amostra = (date.today() - timedelta(days=200))
ano, mes, dia = data_amostra.strftime("%Y"), data_amostra.strftime("%m"), data_amostra.strftime("%d")

url_csv_item = (
    f"{BASE_BULK}/diario/{ano}/{mes}/{dia}/"
    f"comprasGOV-diario-VW_FT_PNCP_COMPRA_ITEM-{ano}-{mes}-{dia}.csv"
)
print("Baixando:", url_csv_item)

resp_csv = requests.get(url_csv_item, timeout=60)
print("Status:", resp_csv.status_code, " | Tamanho:", len(resp_csv.content) / 1e6, "MB")

if resp_csv.status_code == 200:
    local_path = DATA_DIR / f"amostra_item_{ano}{mes}{dia}.csv"
    local_path.write_bytes(resp_csv.content)
    print("Salvo em:", local_path)
else:
    print("Não encontrado nessa data — tente outro dia (ex.: subtraia mais dias) ou verifique o padrão de URL no manual.")


Baixando: https://repositorio.dados.gov.br/seges/comprasgov/diario/2026/01/22/comprasGOV-diario-VW_FT_PNCP_COMPRA_ITEM-2026-01-22.csv
Status: 200  | Tamanho: 12.149759 MB
Salvo em: ..\data\discovery\amostra_item_20260122.csv


In [8]:
# Leitura e inspeção do CSV bulk baixado
# Testamos ; e , como separador, e como o pandas interpreta os campos de valor
try:
    df_csv_raw = pd.read_csv(local_path, sep=";", encoding="utf-8", low_memory=False, nrows=5000)
    print("Lido com sep=';'")
except Exception as e:
    print("Falhou com sep=';':", e)
    df_csv_raw = pd.read_csv(local_path, sep=",", encoding="utf-8", low_memory=False, nrows=5000)
    print("Lido com sep=','")

print("\nShape (amostra de até 5000 linhas):", df_csv_raw.shape)
print("\nColunas:")
print(list(df_csv_raw.columns))
display(df_csv_raw.head(10))

# Foco no problema relatado: os campos de valor têm casas decimais coerentes?
colunas_valor = [c for c in df_csv_raw.columns if "valor" in c.lower()]
print("\nColunas de valor encontradas:", colunas_valor)
for c in colunas_valor:
    print(f"\n{c} — amostra de valores brutos (como veio do CSV, sem conversão):")
    print(df_csv_raw[c].head(10).tolist())
    print(f"{c} — dtype inferido pelo pandas:", df_csv_raw[c].dtype)


Falhou com sep=';': Error tokenizing data. C error: Expected 1 fields in line 595, saw 5

Lido com sep=','

Shape (amostra de até 5000 linhas): (5000, 56)

Colunas:
['srk_pncp_item_compra', 'cod_compra', 'cod_item_compra', 'codigo_classe', 'codigo_grupo', 'numero_item_pncp', 'numero_grupo', 'numero_item_compra', 'descricao_detalhada', 'descricao_resumida', 'material_ou_servico', 'material_ou_servico_nome', 'valor_unitario_estimado', 'valor_total', 'quantidade', 'unidade_medida', 'orcamento_sigiloso', 'item_categoria_id_pncp', 'item_categoria_nome', 'patrimonio', 'codigo_registro_imobiliario', 'criterio_julgamento_id_pncp', 'criterio_julgamento_nome', 'situacao_compra_item', 'situacao_compra_item_nome', 'tipo_beneficio', 'tipo_beneficio_nome', 'incentivo_produtivo_basico', 'data_inclusao_pncp', 'data_atualizacao_pncp', 'tem_resultado', 'cod_item_catalogo', 'ID_contratacao_PNCP', 'cod_fornecedor', 'nome_fornecedor', 'data_resultado', 'valor_total_resultado', 'valor_unitario_resultado', '

,srk_pncp_item_compra,cod_compra,cod_item_compra,codigo_classe,codigo_grupo,numero_item_pncp,numero_grupo,numero_item_compra,descricao_detalhada,descricao_resumida,material_ou_servico,material_ou_servico_nome,valor_unitario_estimado,valor_total,quantidade,unidade_medida,orcamento_sigiloso,item_categoria_id_pncp,item_categoria_nome,patrimonio,codigo_registro_imobiliario,criterio_julgamento_id_pncp,criterio_julgamento_nome,situacao_compra_item,situacao_compra_item_nome,tipo_beneficio,tipo_beneficio_nome,incentivo_produtivo_basico,data_inclusao_pncp,data_atualizacao_pncp,tem_resultado,cod_item_catalogo,ID_contratacao_PNCP,cod_fornecedor,nome_fornecedor,data_resultado,valor_total_resultado,valor_unitario_resultado,quantidade_resultado,id_compra,id_compra_item,ano_compra,sequencial_compra,orgao_entidade_cnpj,unidade_orgao_codigo_unidade,margem_preferencia_normal,percentual_margem_preferencia_normal,margem_preferencia_adicional,percentual_margem_preferencia_adicional,codigo_NCM,descricao_NCM,numero_controle_PNCP_compra,TXT_LINK_MATERIAL,TXT_LINK_CLASSE,TXT_LINK_SERVICO,TXT_LINK_GRUPO
0,41872588,9346936,39271921,NaN,822.0,5,0,5,"NAVIO LIGEIRO, NOME: NAVIO LIGEIRO",Auditoria em área de processamento de dados,S,Serviço,69.90,38095.50,545.0,UNIDADE,False,3,Informática (TIC),NaN,NaN,1,Menor preço,5,Fracassado,1,Participação exclusiva para ME/EPP,False,2025-09-19 07:17:12,2026-01-22 14:48:36,NaN,736.0,10572022000180-1-000890/2025,NaN,NaN,NaN,NaN,NaN,NaN,92615005904352025,9261500590435202500005,2025,890,10572022000180,926150,False,NaN,False,NaN,NaN,NaN,10572022000180-1-000890/2025,NaN,NaN,https://dadosabertos.compras.gov.br/modulo-ser...,https://dadosabertos.compras.gov.br/modulo-ser...
1,42030502,9864826,45938118,NaN,542.0,1,0,1,"BARRA VIDRO, NOME: BARRA DE VIDRO",Obras Civis de Pavimentação Asfáltica,S,Serviço,20371701.92,20371701.92,1.0,METRO QUADRADO,False,3,Informática (TIC),NaN,NaN,1,Menor preço,2,Homologado,5,Não se aplica,False,2025-12-16 07:00:12,2026-01-22 09:15:08,True,1406.0,07954480000179-1-025178/2025,1.660807e+13,FFX ENGENHARIA & SERVICOS LTDA,2026-01-22,15885900.00,15885900.00,1.0,94300103951152025,9430010395115202500001,2025,25178,7954480000179,943001,False,NaN,False,NaN,NaN,NaN,07954480000179-1-025178/2025,NaN,NaN,https://dadosabertos.compras.gov.br/modulo-ser...,https://dadosabertos.compras.gov.br/modulo-ser...
2,41925356,9847110,45709737,NaN,631.0,42,0,42,"RETENTOR DE ÓLEO, NOME: RETENTOR DE OLEO",Reserva em Hotéis Nacionais e Internacionais,S,Serviço,250.06,5001.20,20.0,UNIDADE,False,3,Informática (TIC),NaN,NaN,1,Menor preço,2,Homologado,1,Participação exclusiva para ME/EPP,False,2025-12-12 07:13:56,2026-01-22 14:38:02,True,9946.0,75771253000168-1-000383/2025,4.212972e+13,W V SERVICOS LTDA,2026-01-22,4980.00,249.00,20.0,98742505901042025,9874250590104202500042,2025,383,75771253000168,987425,False,NaN,False,NaN,NaN,NaN,75771253000168-1-000383/2025,NaN,NaN,https://dadosabertos.compras.gov.br/modulo-ser...,https://dadosabertos.compras.gov.br/modulo-ser...
3,41891264,9931319,46860327,NaN,859.0,1,0,1,"TIRISTOR POTÊNCIA, NOME: TIRISTOR SEMICONDUTOR",Prestação de Serviços de Copeiragem,S,Serviço,63445.32,126890.64,2.0,UNIDADE,False,3,Informática (TIC),NaN,NaN,1,Menor preço,2,Homologado,5,Não se aplica,False,2025-12-31 07:02:05,2026-01-22 17:08:55,True,14397.0,32901688000177-1-000030/2025,3.375601e+13,FALLCON SERVICE LTDA,2026-01-22,109208.76,54604.38,2.0,34404105900062025,3440410590006202500001,2025,30,32901688000177,344041,False,NaN,False,NaN,NaN,NaN,32901688000177-1-000030/2025,NaN,NaN,https://dadosabertos.compras.gov.br/modulo-ser...,https://dadosabertos.compras.gov.br/modulo-ser...
4,41926198,9931319,46860330,NaN,859.0,4,0,4,"TIRISTOR POTÊNCIA, NOME: TIRISTOR SEMICONDUTOR",Prestação de Serviços de Copeiragem,S,Serviço,59456.35,59456.35,1.0,UNIDADE,False,3,Informática (TIC),NaN,NaN,1,Menor preço,2,Homologado,5,Não se aplica,False,2025-12-31 07:02:05,2026-01-22 17:08:55,True,14397.0,32901688000177-1-000030/2025,3.375601e+13


Colunas de valor encontradas: ['valor_unitario_estimado', 'valor_total', 'valor_total_resultado', 'valor_unitario_resultado']

valor_unitario_estimado — amostra de valores brutos (como veio do CSV, sem conversão):
[69.9, 20371701.92, 250.06, 63445.32, 59456.35, 101886.43, 80075.88, 10536.48, 0.0, 0.0]
valor_unitario_estimado — dtype inferido pelo pandas: float64

valor_total — amostra de valores brutos (como veio do CSV, sem conversão):
[38095.5, 20371701.92, 5001.2, 126890.64, 59456.35, 2037728.6, 1601517.6, 758626.56, 0.0, 0.0]
valor_total — dtype inferido pelo pandas: float64

valor_total_resultado — amostra de valores brutos (como veio do CSV, sem conversão):
[nan, 15885900.0, 4980.0, 109208.76, 51570.06, 1537449.6, 1265380.8, 636041.52, nan, nan]
valor_total_resultado — dtype inferido pelo pandas: float64

valor_unitario_resultado — amostra de valores brutos (como veio do CSV, sem conversão):
[nan, 15885900.0, 249.0, 54604.38, 51570.06, 76872.48, 63269.04, 8833.91, nan, nan]
valo

**Achado confirmado:** o separador real do CSV bulk é vírgula (`,`), não
ponto e vírgula. Isso derrubou a Seção 7 original (que usava `sep=";"` fixo)
— corrigido abaixo, junto com a troca para DuckDB.


## Seção 5 — Data Quality: preço, quantidade, missing, duplicatas

Aplicado sobre a amostra do CSV bulk (Seção 4), por ter volume maior que as
chamadas de API. Testa as regras propostas na Fase 0 (Seção 11 do brief original).


In [9]:
def perfil_qualidade(df, nome_df):
    print(f"===== Perfil de qualidade: {nome_df} =====")
    print("Linhas:", len(df))
    print("\n% de missing por coluna (top 15):")
    display((df.isna().mean() * 100).sort_values(ascending=False).head(15))
    return df

if 'df_csv_raw' in dir():
    _ = perfil_qualidade(df_csv_raw, "amostra bulk CSV — itens")

# Ajuste os nomes de coluna reais assim que a Seção 4 confirmar o schema do CSV
# (os nomes no JSON da API usam camelCase; os nomes no CSV podem vir em snake_case)


===== Perfil de qualidade: amostra bulk CSV — itens =====
Linhas: 5000

% de missing por coluna (top 15):


patrimonio                                 100.00
codigo_registro_imobiliario                100.00
descricao_NCM                              100.00
percentual_margem_preferencia_adicional     99.82
percentual_margem_preferencia_normal        99.48
TXT_LINK_SERVICO                            91.72
codigo_grupo                                91.72
TXT_LINK_GRUPO                              91.72
codigo_NCM                                  85.66
tem_resultado                               37.58
data_resultado                              37.36
nome_fornecedor                             37.36
quantidade_resultado                        37.36
valor_unitario_resultado                    37.36
cod_fornecedor                              37.36
dtype: float64

In [10]:
# Regras de Data Quality propostas na Fase 0 — ajuste os nomes de coluna
# conforme confirmado na Seção 4 antes de rodar esta célula
def checar_regras_dq(df, col_preco_unit, col_qtd, col_preco_total, col_chave):
    n = len(df)
    resultados = {}

    if col_preco_unit in df.columns:
        resultados["preco_unitario_<=_0"] = (pd.to_numeric(df[col_preco_unit], errors="coerce") <= 0).sum()
    if col_qtd in df.columns:
        resultados["quantidade_<=_0"] = (pd.to_numeric(df[col_qtd], errors="coerce") <= 0).sum()
    if col_preco_total in df.columns:
        resultados["preco_total_<_0"] = (pd.to_numeric(df[col_preco_total], errors="coerce") < 0).sum()
    if col_chave in df.columns:
        resultados["chave_duplicada"] = df[col_chave].duplicated().sum()

    for k, v in resultados.items():
        pct = 100 * v / n if n else 0
        print(f"{k}: {v} registros ({pct:.2f}%)")
    return resultados

# Exemplo de chamada — AJUSTAR nomes reais de coluna após ver a Seção 4
# checar_regras_dq(df_csv_raw, col_preco_unit="valor_unitario", col_qtd="quantidade_item",
#                   col_preco_total="valor_total", col_chave="id_item_compra")


## Seção 6 — Qualidade do CATMAT/CATSER (`codItemCatalogo`)

Este é o teste mais importante do notebook. Um usuário relatou no fórum
oficial do portal que este campo vem nulo em qualquer consulta ao endpoint de
itens. Precisamos de um número real, não da anedota de um fórum.


In [11]:
def taxa_null_catmat(df, coluna_catmat):
    if coluna_catmat not in df.columns:
        print(f"Coluna '{coluna_catmat}' não encontrada. Colunas disponíveis: {list(df.columns)}")
        return None
    n = len(df)
    n_null = df[coluna_catmat].isna().sum()
    n_zero = (pd.to_numeric(df[coluna_catmat], errors="coerce") == 0).sum()
    print(f"Total de linhas: {n}")
    print(f"CATMAT/CATSER nulo: {n_null} ({100*n_null/n:.2f}%)")
    print(f"CATMAT/CATSER == 0 (possível placeholder de nulo): {n_zero} ({100*n_zero/n:.2f}%)")
    return {"null": n_null, "zero": n_zero, "total": n}

# Testar tanto na resposta da API (Seção 2) quanto no CSV bulk (Seção 4)
if 'df_itens' in dir() and not df_itens.empty:
    print("--- Via API (Seção 2) ---")
    col_catmat_api = "codItemCatalogo" if "codItemCatalogo" in df_itens.columns else None
    if col_catmat_api:
        taxa_null_catmat(df_itens, col_catmat_api)

if 'df_csv_raw' in dir():
    print("\n--- Via CSV bulk (Seção 4) ---")
    candidatos = [c for c in df_csv_raw.columns if "catalogo" in c.lower() or "catmat" in c.lower() or "catser" in c.lower()]
    print("Colunas candidatas a CATMAT no CSV:", candidatos)
    if candidatos:
        taxa_null_catmat(df_csv_raw, candidatos[0])



--- Via CSV bulk (Seção 4) ---
Colunas candidatas a CATMAT no CSV: ['cod_item_catalogo']
Total de linhas: 5000
CATMAT/CATSER nulo: 1399 (27.98%)
CATMAT/CATSER == 0 (possível placeholder de nulo): 0 (0.00%)


In [12]:
# Se o CATMAT estiver populado o suficiente: mesmo código, descrições/unidades diferentes?
def cardinalidade_por_catmat(df, col_catmat, col_descricao, col_unidade, top_n=15):
    if col_catmat not in df.columns:
        print("Coluna de CATMAT não encontrada nesta amostra.")
        return
    validos = df[df[col_catmat].notna()]
    if validos.empty:
        print("Nenhum registro com CATMAT preenchido nesta amostra — não é possível avaliar cardinalidade.")
        return
    agg = validos.groupby(col_catmat).agg(
        n_descricoes_distintas=(col_descricao, "nunique") if col_descricao in df.columns else (col_catmat, "count"),
        n_unidades_distintas=(col_unidade, "nunique") if col_unidade in df.columns else (col_catmat, "count"),
        n_ocorrencias=(col_catmat, "count"),
    ).sort_values("n_ocorrencias", ascending=False)
    display(agg.head(top_n))
    print(f"\n% de CATMATs com mais de 1 unidade de medida distinta: "
          f"{100 * (agg['n_unidades_distintas'] > 1).mean():.2f}%")

# Ajustar nomes reais de coluna conforme confirmado nas seções anteriores
# cardinalidade_por_catmat(df_csv_raw, col_catmat="codigo_item_catalogo",
#                           col_descricao="descricao_item", col_unidade="unidade_medida")


## Seção 7 — Cobertura temporal (2021-2026)

Conta registros por ano a partir dos arquivos anuais do repositório bulk,
sem baixar o arquivo inteiro na memória (`nrows` + leitura em chunks) — os
arquivos anuais podem ser grandes. Objetivo: visualizar a transição
Lei 8.666 → Lei 14.133 e confirmar se 2022-2025 tem volume suficiente e
consistente para o split temporal proposto na Fase 0.


**Correção:** a função original usava `sep=";"` por padrão, mas a Seção 4
confirmou que o separador real é vírgula — foi isso que quebrou a contagem,
não um problema nos dados. Aproveitando para trocar por DuckDB, que já é a
engine SQL escolhida na Fase 0: lida melhor com CSVs grandes e um pouco
malformados (usamos `ignore_errors=true` para não travar em linhas com
campos de texto livre mal escapados) e não exige carregar o arquivo inteiro
em memória Python.

Rode `pip install duckdb` no terminal se ainda não tiver instalado.

O arquivo de 2026 é o ano corrente e pode estar grande (o teste anterior
travou em ~660MB por `IncompleteRead`) — se travar de novo ou demorar demais,
pode comentar a linha `2026` na lista `anos` e seguir só com 2021-2025, que é
a janela que realmente importa para o split temporal da Fase 0.


In [17]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

anos = [2021, 2022, 2023, 2024, 2025, 2026]
contagens = {}

for ano in anos:
    url_anual = f"{BASE_BULK}/anual/{ano}/comprasGOV-anual-VW_FT_PNCP_COMPRA_ITEM-{ano}.csv"
    print(f"Contando registros de {ano}...")
    try:
        result = con.execute(
            f"""
            SELECT COUNT(*) FROM read_csv_auto('{url_anual}', ignore_errors=true)
            """
        ).fetchone()
        contagens[ano] = result[0]
        print(f"  {ano}: {result[0]:,}")
    except Exception as e:
        print(f"  {ano}: erro — {e}")
        contagens[ano] = None

df_cobertura = pd.DataFrame(list(contagens.items()), columns=["ano", "n_registros_itens"])
display(df_cobertura)

print("\nObservação: 'ignore_errors=true' descarta linhas malformadas em vez de travar a contagem.")
print("Se o número de linhas descartadas for relevante, isso também é um achado de Data Quality —")
print("vale checar rodando sem ignore_errors em uma amostra menor para ver a taxa real de erro.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Contando registros de 2021...
  2021: 28,892
Contando registros de 2022...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2022: 156,285
Contando registros de 2023...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

  2025: 4,736,611
Contando registros de 2026...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2026: 3,268,675


,ano,n_registros_itens
0,2021,28892
1,2022,156285
2,2023,290640
3,2024,1642583
4,2025,4736611
5,2026,3268675



Observação: 'ignore_errors=true' descarta linhas malformadas em vez de travar a contagem.
Se o número de linhas descartadas for relevante, isso também é um achado de Data Quality —
vale checar rodando sem ignore_errors em uma amostra menor para ver a taxa real de erro.


## Seção 8 — Heterogeneidade de unidade de medida

Quantifica o problema central levantado na Fase 0: o mesmo item aparecendo
com unidades de medida diferentes (unidade, caixa, litro, etc.), o que
inviabilizaria comparação de preço sem tratamento prévio.


In [18]:
def distribuicao_unidades_por_item(df, col_item, col_unidade, top_n=20):
    if col_item not in df.columns or col_unidade not in df.columns:
        print(f"Colunas não encontradas: {col_item}, {col_unidade}")
        return
    tab = (
        df.groupby(col_item)[col_unidade]
        .nunique()
        .sort_values(ascending=False)
    )
    print(f"Itens com mais de 1 unidade de medida distinta: {(tab > 1).sum()} de {len(tab)} "
          f"({100 * (tab > 1).mean():.2f}%)")
    display(tab.head(top_n))


print("--- Heterogeneidade de unidade por CATMAT (cod_item_catalogo), quando presente ---")
distribuicao_unidades_por_item(
    df_csv_raw[df_csv_raw["cod_item_catalogo"].notna()],
    col_item="cod_item_catalogo",
    col_unidade="unidade_medida",
)

print("\n--- Heterogeneidade de unidade por descrição resumida (cobre também os ~28% sem CATMAT) ---")
distribuicao_unidades_por_item(
    df_csv_raw,
    col_item="descricao_resumida",
    col_unidade="unidade_medida",
)


--- Heterogeneidade de unidade por CATMAT (cod_item_catalogo), quando presente ---
Itens com mais de 1 unidade de medida distinta: 222 de 2272 (9.77%)


cod_item_catalogo
463699.0    6
459670.0    6
271950.0    5
268236.0    4
464511.0    3
464474.0    3
464514.0    3
419859.0    3
446393.0    3
458910.0    3
458953.0    3
459002.0    3
465332.0    3
3417.0      3
444323.0    3
268504.0    3
268481.0    3
267777.0    3
291893.0    3
292196.0    3
Name: unidade_medida, dtype: int64


--- Heterogeneidade de unidade por descrição resumida (cobre também os ~28% sem CATMAT) ---
Itens com mais de 1 unidade de medida distinta: 281 de 1882 (14.93%)


descricao_resumida
Biscoito                   10
Cola                        8
Tinta Esmalte               6
Pão                         5
Corante                     5
Glicose                     5
Dexametasona                5
Suco                        5
Manteiga                    5
Água Destilada              4
Clorexidina Digluconato     4
Eletroduto                  4
Contraste Radiológico       4
Cimento Odontológico        4
Clipe                       4
Cloreto De Sódio            4
Tempero                     4
Chá Alimentação             4
Álcool Etílico              4
Café                        4
Name: unidade_medida, dtype: int64

## Seção 9 — Grão real da `fact_purchase`

A Fase 0 assumiu grão = item de compra (`idCompraItem`). Testamos aqui se
essa suposição se sustenta: um item pode ter mais de um resultado
homologado (múltiplos fornecedores em Sistema de Registro de Preços, ou
homologação parcial)?


**Ajuste:** a versão original dependia de `df_itens` (Seção 2), que não
retornou dado por causa do 404. Como o CSV bulk já mescla item e resultado
numa linha por `id_compra_item`, testamos o grão diretamente nele: se o mesmo
`id_compra_item` aparecer mais de uma vez, é sinal de múltiplos fornecedores
homologados para o mesmo item (comum em Registro de Preço com mais de um
vencedor) — e nesse caso o grão da `fact_purchase` precisa incluir o
fornecedor, não só o item.


In [19]:
def testar_grao_csv(df, col_item="id_compra_item", col_fornecedor="cod_fornecedor"):
    n = len(df)
    dup_mask = df[col_item].duplicated(keep=False)
    n_dup = df[col_item].duplicated().sum()
    print(f"'{col_item}' duplicado: {n_dup} de {n} linhas ({100*n_dup/n:.2f}%)")

    if n_dup > 0:
        ids_duplicados = df.loc[dup_mask, col_item].unique()[:5]
        amostra = df[df[col_item].isin(ids_duplicados)].sort_values(col_item)
        cols_mostrar = [col_item, col_fornecedor, "valor_unitario_resultado", "quantidade_resultado"]
        cols_mostrar = [c for c in cols_mostrar if c in df.columns]
        print("\nAmostra de linhas duplicadas (para inspecionar se o fornecedor varia):")
        display(amostra[cols_mostrar])
        print("\n>> Se o fornecedor varia entre as duplicatas, o grão correto da fact_purchase")
        print(">> é (id_compra_item x cod_fornecedor), não id_compra_item sozinho.")
    else:
        print(f"\n>> Grão confirmado nesta amostra: 1 linha por '{col_item}'.")
        print(">> fact_purchase pode usar id_compra_item como chave primária.")

testar_grao_csv(df_csv_raw)


'id_compra_item' duplicado: 169 de 5000 linhas (3.38%)

Amostra de linhas duplicadas (para inspecionar se o fornecedor varia):


,id_compra_item,cod_fornecedor,valor_unitario_resultado,quantidade_resultado
25,9261500390061202500001,2.724778e+12,1.269795e+07,1.0
26,9261500390061202500001,5.545366e+12,1.269800e+07,1.0
130,9271610590023202500002,4.751903e+13,9.000000e-03,1.0
137,9271610590023202500002,4.751903e+13,9.000000e-03,1.0
127,9271610590023202500004,7.020300e+11,2.170000e-02,1.0
128,9271610590023202500004,7.020300e+11,2.170000e-02,1.0
131,9271610590023202500005,8.043040e+11,1.304000e-01,1.0
132,9271610590023202500005,8.043040e+11,1.304000e-01,1.0
129,9271610590023202500010,1.163424e+13,1.800000e-03,1.0
133,9271610590023202500010,1.163424e+13,1.800000e-03,1.0



>> Se o fornecedor varia entre as duplicatas, o grão correto da fact_purchase
>> é (id_compra_item x cod_fornecedor), não id_compra_item sozinho.


## Seção 10 — Matriz de campos: confirmado / derivado / não encontrado

Preenchido a partir da documentação oficial (ver mensagem de acompanhamento
deste notebook) e atualizado com o que as Seções 1-9 confirmaram na prática.


In [20]:
matriz_campos = pd.DataFrame([
    {"campo_do_brief": "Identificador da compra", "status": "confirmado", "campo_real": "idCompra / id_compra", "observacao": "presente em Contratações, Resultado e CSV bulk"},
    {"campo_do_brief": "Identificador do item", "status": "confirmado", "campo_real": "idCompraItem / id_compra_item", "observacao": "candidato a grão — ver Seção 9"},
    {"campo_do_brief": "CATMAT/CATSER", "status": "confirmado, 27.98% null", "campo_real": "codItemCatalogo / cod_item_catalogo", "observacao": "número real medido (Seção 6), não a anedota do fórum"},
    {"campo_do_brief": "Descrição do item", "status": "confirmado", "campo_real": "descricao_resumida / descricao_detalhada", "observacao": "fallback de chave quando CATMAT é nulo"},
    {"campo_do_brief": "Quantidade", "status": "confirmado", "campo_real": "quantidade / quantidade_resultado", "observacao": "usar quantidade_resultado para preço praticado"},
    {"campo_do_brief": "Unidade de medida", "status": "confirmado, texto livre", "campo_real": "unidade_medida", "observacao": "heterogeneidade a validar na Seção 8"},
    {"campo_do_brief": "Preço unitário", "status": "confirmado, 2 variantes no CSV", "campo_real": "valor_unitario_resultado (preferir)", "observacao": "valor_unitario_estimado é pré-licitação, não usar como preço observado"},
    {"campo_do_brief": "Preço total", "status": "confirmado", "campo_real": "valor_total / valor_total_resultado", "observacao": ""},
    {"campo_do_brief": "Fornecedor/CNPJ", "status": "confirmado", "campo_real": "cod_fornecedor / nome_fornecedor (CSV); niFornecedor (API Resultado)", "observacao": "validar se cod_fornecedor no CSV é de fato CNPJ formatado"},
    {"campo_do_brief": "Órgão comprador (CNPJ)", "status": "confirmado direto", "campo_real": "orgao_entidade_cnpj", "observacao": "presente tanto no CSV de item quanto na API Resultado"},
    {"campo_do_brief": "Órgão comprador (razão social)", "status": "derivado via join", "campo_real": "orgaoEntidadeRazaoSocial", "observacao": "só em Contratações (VW_FT_PNCP_COMPRA) — join por id_compra"},
    {"campo_do_brief": "UF/região", "status": "depende da fonte", "campo_real": "unidadeOrgaoUfSigla", "observacao": "direto na API Resultado; AUSENTE no CSV bulk de item — join com VW_FT_PNCP_COMPRA necessário"},
    {"campo_do_brief": "Modalidade", "status": "derivado via join", "campo_real": "codigoModalidade/modalidadeNome", "observacao": "ausente no CSV de item — join com VW_FT_PNCP_COMPRA necessário"},
    {"campo_do_brief": "Data da compra", "status": "decisão de design necessária", "campo_real": "dataPublicacaoPncp vs data_resultado", "observacao": "usar data_resultado para preço praticado"},
    {"campo_do_brief": "Status de conclusão", "status": "confirmado (achado novo)", "campo_real": "tem_resultado", "observacao": "~37% da amostra diária ainda sem resultado — filtrar tem_resultado=1 no Silver"},
])
display(matriz_campos)


,campo_do_brief,status,campo_real,observacao
0,Identificador da compra,confirmado,idCompra / id_compra,"presente em Contratações, Resultado e CSV bulk"
1,Identificador do item,confirmado,idCompraItem / id_compra_item,candidato a grão — ver Seção 9
2,CATMAT/CATSER,"confirmado, 27.98% null",codItemCatalogo / cod_item_catalogo,"número real medido (Seção 6), não a anedota do..."
3,Descrição do item,confirmado,descricao_resumida / descricao_detalhada,fallback de chave quando CATMAT é nulo
4,Quantidade,confirmado,quantidade / quantidade_resultado,usar quantidade_resultado para preço praticado
5,Unidade de medida,"confirmado, texto livre",unidade_medida,heterogeneidade a validar na Seção 8
6,Preço unitário,"confirmado, 2 variantes no CSV",valor_unitario_resultado (preferir),"valor_unitario_estimado é pré-licitação, não u..."
7,Preço total,confirmado,valor_total / valor_total_resultado,
8,Fornecedor/CNPJ,confirmado,cod_fornecedor / nome_fornecedor (CSV); niForn...,validar se cod_fornecedor no CSV é de fato CNP...
9,Órgão comprador (CNPJ),confirmado direto,orgao_entidade_cnpj,presente tanto no CSV de item quanto na API Re...


## Seção 11 — Checklist de decisão: GO / GO COM AJUSTES / NO-GO

Preenchido com o que já temos de dado real. Três perguntas ainda dependem de
rodar as células corrigidas das Seções 7, 8 e 9.

| Pergunta | Resposta |
|---|---|
| Taxa de null do CATMAT (Seção 6) foi menor que ~30%? | **Sim — 27,98% medido em amostra real (5.000 linhas, 22/01/2026).** Refuta o relato do fórum de que o campo "vem nulo em qualquer consulta". |
| A cardinalidade de unidade por item (Seção 8) é administrável ou generalizada? | *Pendente — rodar Seção 8 corrigida* |
| A cobertura temporal 2022-2025 (Seção 7) é consistente, ou há gap na transição de lei? | *Pendente — rodar Seção 7 corrigida (DuckDB)* |
| O grão real (Seção 9) é 1:1 (id_compra_item) ou 1:N (item x fornecedor)? | *Pendente — rodar Seção 9 corrigida* |
| As regras de Data Quality (Seção 5) mostraram taxa de erro aceitável? | **Parcial: ~37% dos registros não têm resultado ainda (`tem_resultado=False`) — isso é censura esperada (compra em andamento), não erro. Ainda não testamos preço/quantidade <= 0 nem duplicatas de chave — rodar a Célula da Seção 5 com `checar_regras_dq(df_csv_raw, col_preco_unit="valor_unitario_estimado", col_qtd="quantidade", col_preco_total="valor_total", col_chave="id_compra_item")`.** |
| Os arquivos bulk CSV (Seção 4) têm volume/formato viáveis para o Bronze proposto? | **Sim — arquivo diário de item teve 12,15 MB, 56 colunas, separador vírgula confirmado, valores decimais consistentes. Volume anual ainda não confirmado (pendente Seção 7).** |

**Atualização da minha recomendação preliminar:** com CATMAT em ~28% de null
(não ~100% como o relato do fórum sugeria), a estratégia de usar
`descricao_resumida` como chave primária de item deixa de ser obrigatória —
pode ser um **fallback** para o ~28% sem CATMAT, em vez de substituir CATMAT
inteiramente. Isso é uma boa notícia para a qualidade do benchmarking de preço
por item, mas ainda depende do que a Seção 8 mostrar sobre heterogeneidade de
unidade dentro do mesmo CATMAT — se a maioria dos CATMATs tiver uma única
unidade de medida, a comparabilidade fica mais simples do que o pior cenário
da Fase 0.

**Próximo passo:** rode as três células corrigidas (Seções 7, 8, 9) e a
regra de Data Quality pendente na Seção 5, salve o notebook e traga de volta.
Com os três números que faltam, fechamos o GO / GO COM AJUSTES / NO-GO.
